# Scraping

In [11]:
import pandas as pd
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from utils.constants import *
from utils.scraping import *

In [ ]:
TYPE = 'mencabut'
LAW = 'uu'
RELATIONSHIP_TYPE = TYPE.upper()

web = f"https://peraturan.go.id/{TYPE}/{LAW}"


In [54]:
service = Service(executable_path=CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service)
driver.get(web)

In [55]:
# Find the table rows using XPath
rows = driver.find_elements(By.XPATH, "//table/tbody/tr")

# Initialize a set to hold unique rows and a list to collect the data
unique_rows = set()
data = []

for row in rows:
    # Extract data from each column
    no = row.find_element(By.XPATH, "./th").text
    peraturan_element = row.find_element(By.XPATH, "./td[1]/a")
    peraturan = peraturan_element.text
    peraturan_link = peraturan_element.get_attribute("href")
    status = row.find_element(By.XPATH, "./td[2]").text
    related_peraturan_elements = row.find_elements(By.XPATH, "./td[3]/a")

    # Create unique rows by iterating over related peraturan
    for related_peraturan_element in related_peraturan_elements:
        related_peraturan = related_peraturan_element.text
        related_peraturan_link = related_peraturan_element.get_attribute("href")
        
        # Create a tuple representing the row with both links
        row_tuple = (peraturan, status, related_peraturan, peraturan_link, related_peraturan_link)
        
        # Add the tuple to the set to ensure uniqueness
        if row_tuple not in unique_rows:
            unique_rows.add(row_tuple)
            data.append(row_tuple)

# Convert the list of tuples to a DataFrame
df = pd.DataFrame(data, columns=["PERATURAN", "STATUS", "RELATED_PERATURAN", "PERATURAN_LINK", "RELATED_PERATURAN_LINK"])

In [65]:
df2 = df

In [66]:
# Define the exact key mappings
key_mapping = {
    "Jenis/Bentuk Peraturan": "jenis_peraturan",
    "Pemrakarsa": "pemrakarsa",
    "Nomor": "nomor",
    "Tahun": "tahun",
    "Tentang": "tentang",
    "Tempat Penetapan": "tempat_penetapan",
    "Status": "status",
}

# Dictionary to cache fetched metadata
metadata_cache = {}

title_div_xpath = '//div[@class="detail_title_1"]'
table_xpath = '//table[@class="table caption-top"]'
row_xpath = './/tr'
row_q_xpath = './/th[@style="width: 30%;"]'
row_a_xpath = './/td'

SKIP_COLS = [
    "pejabat_yang_menetapkan", "tahun_pengundangan", "nomor_pengundangan",
    "nomor_tambahan", "tanggal_pengundangan", "pejabat_pengundangan"
]

# Function to fetch metadata for a given link
def fetch_metadata(link):
    # Check if the link is already in the cache
    if link in metadata_cache:
        return metadata_cache[link]

    driver.get(link)
    WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, table_xpath)))

    reg_detail = {}
    title_element = driver.find_element(By.XPATH, title_div_xpath)
    reg_detail["title"] = title_element.text

    tables = driver.find_elements(By.XPATH, table_xpath)

    for table in tables:
        rows = table.find_elements(By.XPATH, row_xpath)
        for row_element in rows:
            row_q = row_element.find_element(By.XPATH, row_q_xpath)
            row_q_text = row_q.text.strip()

            if row_q_text in SKIP_COLS:
                continue

            # Capture the text content properly even if there are nested elements
            row_a = row_element.find_element(By.XPATH, row_a_xpath)
            reg_detail[key_mapping.get(row_q_text, row_q_text)] = row_a.get_attribute('textContent').strip()

    # Store the fetched metadata in the cache
    metadata_cache[link] = reg_detail

    return reg_detail


# List to hold the combined metadata for each row
combined_metadata = []

for index, row in df2.iterrows():
    print(f"Processing row {index + 1}/{len(df2)}")

    # Fetch metadata for PERATURAN_LINK
    peraturan_metadata = fetch_metadata(row['PERATURAN_LINK'])

    # Fetch metadata for RELATED_PERATURAN_LINK
    related_peraturan_metadata = fetch_metadata(row['RELATED_PERATURAN_LINK'])

    # Combine both metadata into a single dictionary
    combined_metadata_row = {**row.to_dict()}
    combined_metadata_row.update({
        f"PERATURAN_{key}": peraturan_metadata.get(key, "") for key in key_mapping.values()
    })
    combined_metadata_row.update({
        f"RELATED_{key}": related_peraturan_metadata.get(key, "") for key in key_mapping.values()
    })

    combined_metadata.append(combined_metadata_row)

# Convert the list of combined metadata dictionaries to a DataFrame
df_combined = pd.DataFrame(combined_metadata)

Processing row 1/532
Processing row 2/532
Processing row 3/532
Processing row 4/532
Processing row 5/532
Processing row 6/532
Processing row 7/532
Processing row 8/532
Processing row 9/532
Processing row 10/532
Processing row 11/532
Processing row 12/532
Processing row 13/532
Processing row 14/532
Processing row 15/532
Processing row 16/532
Processing row 17/532
Processing row 18/532
Processing row 19/532
Processing row 20/532
Processing row 21/532
Processing row 22/532
Processing row 23/532
Processing row 24/532
Processing row 25/532
Processing row 26/532
Processing row 27/532
Processing row 28/532
Processing row 29/532
Processing row 30/532
Processing row 31/532
Processing row 32/532
Processing row 33/532
Processing row 34/532
Processing row 35/532
Processing row 36/532
Processing row 37/532
Processing row 38/532
Processing row 39/532
Processing row 40/532
Processing row 41/532
Processing row 42/532
Processing row 43/532
Processing row 44/532
Processing row 45/532
Processing row 46/5

In [69]:
df_combined.head(1)

,PERATURAN,STATUS,RELATED_PERATURAN,PERATURAN_LINK,RELATED_PERATURAN_LINK,PERATURAN_jenis_peraturan,PERATURAN_pemrakarsa,PERATURAN_nomor,PERATURAN_tahun,PERATURAN_tentang,PERATURAN_tempat_penetapan,PERATURAN_status,RELATED_jenis_peraturan,RELATED_pemrakarsa,RELATED_nomor,RELATED_tahun,RELATED_tentang,RELATED_tempat_penetapan,RELATED_status
0,uu-no-13-tahun-2003,Mencabut,uu-no-25-tahun-1997,https://peraturan.go.id/id/uu-no-13-tahun-2003,https://peraturan.go.id/id/uu-no-25-tahun-1997,UNDANG-UNDANG,PEMERINTAH PUSAT,13,2003,KETENAGAKERJAAN,Jakarta,Berlaku,UNDANG-UNDANG,PEMERINTAH PUSAT,25,1997,KETENAGAKERJAAN,Jakarta,Tidak Berlaku Dicabut Oleh :Undang-Undang Nomor 13 Tahun 2003 Tentang KetenagakerjaanUndang-Undang Nomor 13 Tahun 2003 Tentang KetenagakerjaanUndang-Undang Nomor 13 Tahun 2003 Tentang Ketenagakerjaan


In [71]:
df_combined.to_csv(f"csv/{LAW}/{TYPE}.csv", index=False)

# Inserting

In [91]:
from py2neo import Graph
from utils.config import get_config

config = get_config()

graph = Graph(config['neo4j_uri'], name=config['neo4j_username'], password=config['neo4j_password'])

In [92]:
graph

Graph('neo4j+s://0b76488f.databases.neo4j.io:7687', name='neo4j')

In [93]:
from py2neo import Node, Relationship

nodes = {}

# Function to create nodes
def create_node(label, properties):
    node_id = properties['id']
    if node_id not in nodes:
        node = Node(label, **properties)
        graph.merge(node, label, "id")
        nodes[node_id] = node
    return nodes[node_id]

# Function to create relationships
def create_relationship(source_id, target_id, relationship_type):
    source_node = nodes.get(source_id)
    target_node = nodes.get(target_id)
    if source_node and target_node:
        relationship = Relationship(source_node, relationship_type, target_node)
        graph.merge(relationship)

# Create the reverse relationship
def create_reverse_relationship(source_id, target_id, relationship_type):
    if relationship_type == "MENCABUT":
        reverse_relationship_type = "DICABUT"
    elif relationship_type == "MENGUBAH":
        reverse_relationship_type = "DIUBAH"
    else:
        raise ValueError(f"Unknown relationship type: {relationship_type}")
    
    create_relationship(target_id, source_id, reverse_relationship_type)


# Iterate over the DataFrame and create nodes and relationships
for index, row in df_combined.iterrows():
    # Create a node for PERATURAN
    peraturan_properties = {
        "id": row['PERATURAN_LINK'],
        "nomor": row['PERATURAN_nomor'],
        "jenis_peraturan": row['PERATURAN_jenis_peraturan'],
        "pemrakarsa": row['PERATURAN_pemrakarsa'],
        "tahun": row['PERATURAN_tahun'],
        "tentang": row['PERATURAN_tentang'],
        "tempat_penetapan": row['PERATURAN_tempat_penetapan'],
        "status": row['PERATURAN_status'],
    }
    peraturan_node = create_node("Peraturan", peraturan_properties)

    # Create a node for RELATED_PERATURAN
    related_properties = {
        "id": row['RELATED_PERATURAN_LINK'],
        "nomor": row['RELATED_nomor'],
        "jenis_peraturan": row['RELATED_jenis_peraturan'],
        "pemrakarsa": row['RELATED_pemrakarsa'],
        "tahun": row['RELATED_tahun'],
        "tentang": row['RELATED_tentang'],
        "tempat_penetapan": row['RELATED_tempat_penetapan'],
        "status": row['RELATED_status'],
    }
    related_node = create_node("Peraturan", related_properties)

    create_relationship(row['PERATURAN_LINK'], row['RELATED_PERATURAN_LINK'], RELATIONSHIP_TYPE)
    create_reverse_relationship(row['PERATURAN_LINK'], row['RELATED_PERATURAN_LINK'], RELATIONSHIP_TYPE)
